# ECG Signal Quality — Exploratory Data Analysis

PhysioNet 2021 has **no signal-quality labels** (only diagnoses). We treat every real
recording as a **Clean** source and synthesise the **Noisy** and **Artifact** classes
via controlled corruption (see `src/dataset.py`).

Run `python run_download.py` and `python build_manifest.py` from the project root first.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('../data/records.csv')
print(f'{len(df)} clean source records')
print('Sampling frequencies:', df['fs'].value_counts().to_dict())
df['sig_len'].describe()

## One record under each synthetic quality class

Same underlying ECG, corrupted three different ways.

In [ ]:
from src.dataset import ECGQualityDataset, CLASS_NAMES, _zscore

ds = ECGQualityDataset(df, train=False)
row = df.iloc[0]
clean = ds._load(row['path'])            # [12, 5000], z-scored
COLORS = ['#2ecc71', '#f39c12', '#e74c3c']

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
for cls, ax in enumerate(axes):
    rng = np.random.default_rng(cls)
    sig = _zscore(ds._corrupt(clean.copy(), cls, rng))
    ax.plot(sig[1, :1500], lw=0.8, color=COLORS[cls])   # Lead II, first 3s
    ax.set_title(f'Lead II — {CLASS_NAMES[cls]}', fontsize=12)
    ax.set_ylabel('z-score')
axes[-1].set_xlabel('Sample')
plt.tight_layout()
plt.savefig('../outputs/example_signals.png', dpi=150)
plt.show()

## Class balance is by construction

Validation assigns classes deterministically as `idx % 3`, so the three classes are
exactly balanced — no resampling needed.

In [ ]:
from collections import Counter
labels = [ds[i][1] for i in range(60)]
print('Label counts over first 60 val items:', dict(sorted(Counter(labels).items())))
print('(0=Clean, 1=Noisy, 2=Artifact)')